# Import

In [0]:
from pyspark.sql.functions import *

# Read Bronze Activity

In [0]:
activity_bronze_path = "abfss://customer360@stcustomers360dev01.dfs.core.windows.net/bronze/api/customer_activity.json"

df_activity = (
    spark.read
    .option("multiline", "true")
    .json(activity_bronze_path)
)

display(df_activity)

# Check rows, schema

In [0]:
print("Bronze activity count:", df_activity.count())
df_activity.printSchema()

# Clean Activity

In [0]:
df_activity_clean = (
    df_activity
    .dropDuplicates()
    .dropDuplicates(["activity_id"])
    .filter(col("activity_id").isNotNull())
    .filter(col("customer_id").isNotNull())
    .withColumn("activity_id", upper(trim(col("activity_id"))))
    .withColumn("customer_id", col("customer_id").cast("int"))
    .withColumn(
        "activity_type",
        trim(col("activity_type"))
    )
    .withColumn(
        "activity_timestamp",
        to_timestamp(col("activity_timestamp"))
    )
    .withColumn("device", trim(col("device")))
    .withColumn("page", trim(col("page")))
)

In [0]:
df_activity_clean = (
    df_activity_clean
    .withColumn(
        "product_id",
        upper(trim(col("product_id")))
    )
    .withColumn(
        "order_id",
        col("order_id").cast("int")
    )
)

# Load Silver Customers, products, Orders

In [0]:
customers_silver_path = "abfss://customer360@stcustomers360dev01.dfs.core.windows.net/silver/customers/"

df_customers = (
    spark.read
    .format("delta")
    .load(customers_silver_path)
)

products_silver_path = "abfss://customer360@stcustomers360dev01.dfs.core.windows.net/silver/products/"

df_products = (
    spark.read
    .format("delta")
    .load(products_silver_path)
)

orders_silver_path = "abfss://customer360@stcustomers360dev01.dfs.core.windows.net/silver/orders/"

df_orders = (
    spark.read
    .format("delta")
    .load(orders_silver_path)
)

# Validate Customers Ids, Products IDs, Purchase Orders


In [0]:
invalid_customers = df_activity_clean.join(
    df_customers,
    on="customer_id",
    how="left_anti"
)

print(
    "Invalid customer references:",
    invalid_customers.count()
)

display(invalid_customers)

invalid_products = (
    df_activity_clean
    .filter(col("product_id").isNotNull())
    .join(
        df_products,
        on="product_id",
        how="left_anti"
    )
)

print(
    "Invalid product references:",
    invalid_products.count()
)

display(invalid_products)

purchase_activity = df_activity_clean.filter(
    col("activity_type") == "Purchase"
)

invalid_orders = purchase_activity.join(
    df_orders,
    on="order_id",
    how="left_anti"
)

print(
    "Invalid purchase order references:",
    invalid_orders.count()
)

display(invalid_orders)

In [0]:
purchase_check = (
    purchase_activity.alias("a")
    .join(
        df_orders.select(
            "order_id",
            "customer_id",
            "product_id"
        ).alias("o"),
        col("a.order_id") == col("o.order_id"),
        "left"
    )
)
# check mismatches
invalid_purchase_relationships = purchase_check.filter(
    col("o.order_id").isNull() |
    (col("a.customer_id") != col("o.customer_id")) |
    (col("a.product_id") != col("o.product_id"))
)

print(
    "Invalid purchase relationships:",
    invalid_purchase_relationships.count()
)

display(invalid_purchase_relationships)

# Write Activity to Silver

In [0]:
activity_silver_path = "abfss://customer360@stcustomers360dev01.dfs.core.windows.net/silver/customer_activity/"

(
    df_activity_clean.write
    .format("delta")
    .mode("overwrite")
    .save(activity_silver_path)
)

# Verify Silver

In [0]:
df_activity_silver = (
    spark.read
    .format("delta")
    .load(activity_silver_path)
)

display(df_activity_silver)

print(
    "Silver activity count:",
    df_activity_silver.count()
)